<a href="https://colab.research.google.com/github/Nolo-GH/Pytorch-Course-1/blob/main/CNN1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Covert MNIST image files into a tensor of 4-Dimensions (# of images, height, width, color channels)

transfrom = transforms.ToTensor()


In [ ]:
# Train Data
train_data = datasets.MNIST(root='/cnn_data', train=True, download=True, transform=transfrom)

100%|██████████| 9.91M/9.91M [00:00<00:00, 127MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 14.1MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 123MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.25MB/s]


In [ ]:
# Test data
test_data = datasets.MNIST(root='/cnn_data', train=False, download=True, transform=transfrom)

In [ ]:
# Create a small batch size fro images... let's say 10
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [ ]:
# Define our CNN Model
# Describe convolutional layer and what it's doign ( 2 convolutional layers)
# Example before the real one
conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1)

In [ ]:
# Grab 1 MNIST record
for i, (X_train, y_train) in enumerate(train_data):
    break

In [ ]:
X_train.shape

torch.Size([1, 28, 28])

In [ ]:
x = X_train.view(1, 1, 28, 28)

In [ ]:
# PErfom our first convolution
x = F.relu(conv1(x)) # rectified linear unit for our activation function

In [ ]:
x.shape
#Explanation: 1 = single image, 6 = filter we sasked for, 26x26 =

torch.Size([1, 6, 26, 26])

In [ ]:
# pass thru the pooling layer
x = F.max_pool2d(x,2,2) #Kernel of 2 and stride of 2

In [ ]:
# Do our second convollutional layer
x = F.relu(conv2(x))

In [ ]:
# Pooling layer
x = F.max_pool2d(x,2,2)

In [25]:
# Model Class

class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1)
        # Fully connected layer
        self.fc1 = nn.Linear(in_features=16*5*5, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.fc3 = nn.Linear(in_features=84, out_features=10)
    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, 2, 2)
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, 2, 2)

        #Re-view to flatten it out
        X = X.view(-1, 16*5*5)

        # fully connected layers
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)

In [27]:
# Create an instance of our model:
torch.manual_seed(41)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [28]:
# Loss function optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [34]:
#TRAINING
import time
start_time = time.time()

#Create Varaibles to Tracks Things
epochs = 5
train_losses = []
test_losses = []
train_correct = []
test_correct = []

# For loop of Epochs
for i in range(epochs):
    trn_corr = 0
    tst_corr = 0
# Train
    for b, (X_train, y_train) in enumerate(train_loader):
        b+=1 # start our batches at 1

        y_pred = model(X_train) # get predicted values from the predictions
        loss = criterion(y_pred, y_train) # how off are we? Compare the predictions to the correct answers

        predicted = torch.max(y_pred.data, 1)[1] # add up the number of correct predictions.
        batch_corr = (predicted == y_train).sum() #how many we got correct from this specific batch
        trn_corr += batch_corr # Keep track as we go along in traing.

  #Update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

  #Print out some result
        if b%600 == 0:
            print(f'Epoch: {i} Batch: {b} Loss: {loss.item()}')
    train_losses.append(loss)
    train_correct.append(trn_corr)

#Test
    with torch.no_grad():# No gradient so we don't update our weights and biases with test data
        for b, (X_test, y_test) in enumerate(test_loader):
            y_val = model(X_test)
            predicted = torch.max(y_val.data, 1)[1] #Adding up correct predictions
            tst_corr += (predicted == y_test).sum() #T=1 F=0 and sum away

    loss = criterion(y_val, y_test)
    test_losses.append(loss)
    test_correct.append(tst_corr)




current_time = time.time()
total = current_time - start_time
print(f'training took: {total/60} minutes')

Epoch: 0 Batch: 600 Loss: 0.00459330203011632
Epoch: 0 Batch: 1200 Loss: 0.0906534343957901
Epoch: 0 Batch: 1800 Loss: 0.6199954748153687
Epoch: 0 Batch: 2400 Loss: 3.883319004671648e-05
Epoch: 0 Batch: 3000 Loss: 0.01406898908317089
Epoch: 0 Batch: 3600 Loss: 0.0008004838600754738
Epoch: 0 Batch: 4200 Loss: 0.00018577431910671294
Epoch: 0 Batch: 4800 Loss: 0.00794302299618721
Epoch: 0 Batch: 5400 Loss: 0.2053094208240509
Epoch: 0 Batch: 6000 Loss: 6.265459524001926e-05
Epoch: 1 Batch: 600 Loss: 0.00010751785157481208
Epoch: 1 Batch: 1200 Loss: 0.0007110425503924489
Epoch: 1 Batch: 1800 Loss: 0.0011262416373938322
Epoch: 1 Batch: 2400 Loss: 0.3027417063713074
Epoch: 1 Batch: 3000 Loss: 0.0003415062674321234
Epoch: 1 Batch: 3600 Loss: 0.00022364586766343564
Epoch: 1 Batch: 4200 Loss: 0.0011234257835894823
Epoch: 1 Batch: 4800 Loss: 0.00027718886849470437
Epoch: 1 Batch: 5400 Loss: 0.003027329221367836
Epoch: 1 Batch: 6000 Loss: 0.0022578625939786434
Epoch: 2 Batch: 600 Loss: 4.194510620